# Lab 07 - Logistic Regression: Impact of AI on Students

        Source dataset: `Datasets/Impact of AI on Students/ai_student_impact_dataset.csv`

        This notebook adapts the class lab pattern to the student-impact dataset. The source file is never modified.

        ## Lab concepts used

        - Fit class-weighted multinomial logistic regression.
- Evaluate class-level recall and macro metrics.
- Inspect the most influential coefficients.

        Interpretation is predictive and associative only. The Kaggle source does not document how the records were collected or whether they represent observed students.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

def find_dataset():
    relative = Path("Datasets/Impact of AI on Students/ai_student_impact_dataset.csv")
    for start in [Path.cwd(), *Path.cwd().parents]:
        candidate = start / relative
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not locate {relative} from {Path.cwd()}")

DATA_PATH = find_dataset()
df = pd.read_csv(DATA_PATH)
print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns from {DATA_PATH}")

In [ ]:
IDENTIFIER = "Student_ID"
OUTCOMES = ["Post_Semester_GPA", "Skill_Retention_Score", "Burnout_Risk_Level"]
EARLY_RISK_FEATURES = [
    "Major_Category", "Year_of_Study", "Pre_Semester_GPA",
    "Weekly_GenAI_Hours", "Primary_Use_Case",
    "Prompt_Engineering_Skill", "Tool_Diversity", "Paid_Subscription",
    "Traditional_Study_Hours", "Perceived_AI_Dependency",
    "Institutional_Policy",
]
EXPANDED_FEATURES = EARLY_RISK_FEATURES + ["Anxiety_Level_During_Exams"]

df["GPA_Change"] = df["Post_Semester_GPA"] - df["Pre_Semester_GPA"]
df["GPA_Declined"] = (df["GPA_Change"] < 0).astype(int)

assert IDENTIFIER not in EARLY_RISK_FEATURES
assert not set(OUTCOMES).intersection(EARLY_RISK_FEATURES)
print("Leakage policy ready. Primary burnout model excludes anxiety and all post-semester outcomes.")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

def make_preprocessor(frame, scale_numeric=True):
    categorical = [
        column for column in frame.columns
        if pd.api.types.is_string_dtype(frame[column])
        or pd.api.types.is_bool_dtype(frame[column])
    ]
    numeric = [column for column in frame.columns if column not in categorical]
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    return ColumnTransformer(
        transformers=[
            ("numeric", Pipeline(numeric_steps), numeric),
            (
                "categorical",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]),
                categorical,
            ),
        ]
    )

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, ConfusionMatrixDisplay,
    f1_score, balanced_accuracy_score, roc_auc_score,
)

X = df[EARLY_RISK_FEATURES]
y = df["Burnout_Risk_Level"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
model = Pipeline([
    ("preprocess", make_preprocessor(X_train)),
    ("model", LogisticRegression(
        C=1.0, max_iter=1500, class_weight="balanced"
    )),
])
model.fit(X_train, y_train)
prediction = model.predict(X_test)
probability = model.predict_proba(X_test)
print(classification_report(y_test, prediction))
print("Macro F1:", f1_score(y_test, prediction, average="macro"))
print("Balanced accuracy:", balanced_accuracy_score(y_test, prediction))
print("Macro ROC-AUC OVR:", roc_auc_score(
    y_test, probability, multi_class="ovr",
    average="macro", labels=model.classes_
))
ConfusionMatrixDisplay.from_predictions(
    y_test, prediction, normalize="true", cmap="Purples"
)
plt.show()

## Coefficient interpretation

In [ ]:
feature_names = model.named_steps["preprocess"].get_feature_names_out()
coefficients = model.named_steps["model"].coef_
for class_name, class_coefficients in zip(model.classes_, coefficients):
    order = np.argsort(np.abs(class_coefficients))[-10:][::-1]
    display(pd.DataFrame({
        "feature": feature_names[order],
        "coefficient": class_coefficients[order],
        "class": class_name,
    }))

## What was learned from logistic regression

Class weighting and macro metrics prevent the medium-risk class from dominating evaluation. Coefficients describe predictive associations after encoding and scaling; they are not causal effects.